In [1]:
# Directory manipulation for relative notebook imports
import os, sys
import numpy as np
dir2 = os.path.abspath('../')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: sys.path.append(dir1)

# Relative import for relevant methods
from Visualization.plot_util import *
from Visualization.projection_plots import *
from Data.data_load_util import *
from Data.data_stats_util import *
from Simulation.projections_util import *
# from Models.Neat.neat_util import *
from Simulation.projections import *


# Incentive effects on states

In [2]:
# Loading the datasets 
zips_df, state_df, pos_df = make_dataset(granularity='both', remove_outliers=False, load_dir_prefix='../../Data/')
state_sum_df = make_state_dataset(zips_df, None, None, "../../Data/Clean_data/data_by_state_sum.csv", agg="sum")
zips = pd.read_csv('../../Data/Clean_Data/zips.csv')
incentives = [5250, -2000, 0, 4000]
state_codes = state_df['State code'].unique()

state_df_dict = {}
years = 2

# Create a new state_df with updated values based on the incentive modeling
for incentive in incentives:

    placements = pd.read_csv("new_behavior_data_projections/"+str(incentive)+"_projection.csv")
    placements['state_code'] = placements['zip'].apply(lambda x: zips[zips['zip'] == x]['state_code'].values[0])
    placements['energy_added'] = placements['zip'].apply(lambda x: placements[placements['zip'] == x]['panels_'+str(years)+'_years_'+str(incentive)+"_incentive"].values[0] * zips_df[zips_df['region_name'] == x]['yearly_sunlight_kwh_kw_threshold_avg'].values[0])

    state_panel_placements = {state: sum(placements[placements['state_code'] == state]['panels_'+str(years)+'_years_'+str(incentive)+"_incentive"]) for state in state_codes}
    state_eng_added = {state: sum(placements[placements['state_code'] == state]['energy_added']) for state in state_codes}

    # Uncomment this to get zip level dataframes
    # zip_panel_placements = {zip: amount for zip, amount, state in zip(placements['zip'], placements["panels_5_years_"+str(incentive)+"_incentive"], placements['state_code']) if state in state_codes}
    # new_zip_df = updated_df_with_picks(zips_df, zip_panel_placements)

    new_state_df = updated_state_df_with_picks(state_df, state_panel_placements)
    new_state_df['panels_'+str(years)+'_years'] = state_panel_placements.values()
    new_state_df['panels_per_capita'] = new_state_df['panels_'+str(years)+'_years'] / state_sum_df['Total_Population']
    new_state_df['incentive_given_'+str(years)+'_years'] = new_state_df['panels_'+str(years)+'_years'] * incentive
    new_state_df['incentive_given_per_capita'] = new_state_df['incentive_given_'+str(years)+'_years'] / state_sum_df['Total_Population']
    new_state_df['energy_added'] = state_eng_added.values()

    # Save this new state df to a dictionary for later use
    state_df_dict[incentive] = new_state_df

    # Calculating the effect of changing from 5250 incentive
    if incentive != 5250:
        new_state_df['ratio_panels_installs_vs_5250'] = (new_state_df['panels_'+str(years)+'_years'] / state_df_dict[5250]['panels_'+str(years)+'_years']).fillna(0)
        new_state_df['cost_vs_5250'] = state_df_dict[5250]['incentive_given_'+str(years)+'_years'] - new_state_df['incentive_given_'+str(years)+'_years'] 
        new_state_df['energy_change_vs_5250'] =  (new_state_df['energy_added']) - (state_df_dict[5250]['energy_added'])
        new_state_df['energy_change_ratio_vs_5250'] =  ((new_state_df['energy_added']) / (state_df_dict[5250]['energy_added'])).fillna(0)

In [3]:
for incentive in [-2000, 0, 4000]:

    #when incentive is -2000 or 0, set the energy and panel ratio to 0.7 for hawaii

    print(incentive, np.std(state_df_dict[incentive]['energy_change_ratio_vs_5250']) / np.mean(state_df_dict[incentive]['energy_change_ratio_vs_5250']))

    if incentive != 4000:
        state_df_dict[incentive]['energy_change_ratio_vs_5250'].mask(state_df_dict[incentive]['State code'] == 'HI', 0.7, inplace=True)


    # plot_state_map(state_df_dict[incentive], 'panels_'+str(years)+'_years', legend_name="Projected 5-Year Solar Panel Installations by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/panels_added.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'incentive_given_'+str(years)+'_years', legend_name="Projected 5-Year recieved incentives by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/incentive_given.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'incentive_given_per_capita', legend_name="Projected 5-Year recieved incentives by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/incentive_given_per_capita.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'ratio_panels_installs_vs_5250', legend_name="Installations built ratio to 5250 for $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/panel_ratio_to_5250.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'cost_vs_5250', legend_name="Lost incentive dollars lowering to $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/lost_incentive_vs_5250.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'energy_added', legend_name="Change in energy production for $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/energy_added.png", show=False)
    # plot_state_map(state_df_dict[incentive], 'energy_change_vs_5250', legend_name="Change in energy production vs $5250 policy for $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/energy_gen_change_vs_5250.png", show=False)
    plot_state_map(state_df_dict[incentive], 'energy_change_ratio_vs_5250', legend_name="Change in energy production vs $5250 policy for $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/energy_gen_ratio_vs_5250.png", show=False, fill_color="YlGn")



# incentive = 5250
# plot_state_map(state_df_dict[incentive], 'panels_'+str(years)+'_years', legend_name="Projected 5-Year Solar Panel Installations by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/panels_added.png", show=False)
# plot_state_map(state_df_dict[incentive], 'incentive_given_'+str(years)+'_years', legend_name="Projected 5-Year recieved incentives by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/incentive_given.png", show=False)
# plot_state_map(state_df_dict[incentive], 'incentive_given_per_capita', legend_name="Projected 5-Year recieved incentives by State with $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/incentive_given_per_capita.png", show=False)
# plot_state_map(state_df_dict[incentive], 'energy_added', legend_name="Change in energy production for $"+str(incentive)+" Incentive", save_dir="maps/"+str(years)+"_years/"+str(incentive)+"_incentive/energy_added.png", show=False)

-2000 1.4163263348346236


C:\Users\coopt\AppData\Local\Temp\ipykernel_36364\1954962474.py:8: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





0 1.1705691132856213


C:\Users\coopt\AppData\Local\Temp\ipykernel_36364\1954962474.py:8: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





4000 0.5819267211325715
